## Import Libraries

In [ ]:
import os

from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.notebook import tqdm
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
from stream_items import Item
from stream_llm import Stream_Item
from stream_data_tokennizer import Stream_Tokenizer

In [ ]:
load_dotenv(override=True)

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"

## HuggeingFace Login

In [ ]:
hkf_token = os.getenv("HUGGING_KEY")
login(hkf_token)

## Load Dataset From HuggingFace

In [ ]:
username = "KumudithaSilva"
dataset = f"{username}/items_llm_raw_lite"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")

## LLaMA SentencePiece Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

## Base Fine-tunning Prompt

In [ ]:
def format_item(item):
    return f"""
Game: {item.name}
Peak CCU: {item.peakCCU}
Required Age: {item.required_age}
DLC Count: {item.dlcCount}
Supports Windows: {item.supportWindows}
Supports Mac: {item.supportMac}
Supports Linux: {item.supportLinux}
Positive Reviews: {item.positive}
Negative Reviews: {item.negative}
Achievements: {item.achievements}
Recommendations: {item.recommendations}
Release Date: {item.release_year}-{item.release_month}-{item.release_day}
Estimated Owners: {item.min_estimatedOwners} - {item.max_estimatedOwners}
Languages Supported: {item.supported_languages}
Developers: {item.num_developers}
Publishers: {item.num_publishers}
Categories: {item.num_categories}
Genres: {item.num_genres}
Description: {item.small_description}
"""

In [ ]:
def format_stream_item(stream_item):
    return f"""
prompt: {stream_item.prompt},
completion: {stream_item.completion}
"""

In [ ]:
tokenizer_obj = Stream_Tokenizer()

In [ ]:
# token count in the dataset
token_counts = [tokenizer_obj.count_base_tokens(stream_item_data=format_item(item), tokenizer=tokenizer) for item in tqdm(items)]

## Token Distribution

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(token_counts, bins=range(min(token_counts), max(token_counts) + 1), color='lightblue', edgecolor='black')
plt.title('Distribution of Token Counts in the Dataset')
plt.xlabel('Token Count')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

## Token Cutoff

In [ ]:
CUT_OFF = 140

token_cut_off = [count for count in token_counts if count <= CUT_OFF]

print(f"Percentage of items with token count <= {CUT_OFF}: {len(token_cut_off) / len(items) * 100:.2f}%")

## Prompt Generation

In [ ]:
stream_llm_train = []

for item in tqdm(train):
    out = tokenizer_obj.make_prompts(
        stream_item_data=format_item(item),
        tokenizer=tokenizer,
        max_tokens=140,
        do_round=True,
        do_truncate=True,
        price=item.price
    )

    stream_llm_train.append(Stream_Item(id=item.id, prompt=out["prompt"], completion=out["completion"]))

In [ ]:
stream_llm_val = []

for item in tqdm(val):
    out = tokenizer_obj.make_prompts(
        stream_item_data=format_item(item),
        tokenizer=tokenizer,
        max_tokens=140,
        do_round=True,
        do_truncate=True,
        price=item.price
    )

    stream_llm_val.append(Stream_Item(id=item.id, prompt=out["prompt"], completion=out["completion"]))

In [ ]:
stream_llm_test = []

for item in tqdm(test):
    out = tokenizer_obj.make_prompts(
        stream_item_data=format_item(item),
        tokenizer=tokenizer,
        max_tokens=140,
        do_round=True,
        do_truncate=True,
        price=item.price
    )

    stream_llm_test.append(Stream_Item(id=item.id, prompt=out["prompt"], completion=out["completion"]))

In [ ]:
full_stream_llm = stream_llm_train + stream_llm_val + stream_llm_test

In [ ]:
# token count in the dataset
token_counts_full = [tokenizer_obj.count_base_tokens(stream_item_data=format_stream_item(stream_item), tokenizer=tokenizer) for stream_item in tqdm(full_stream_llm)]

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(token_counts_full, bins=range(min(token_counts_full), max(token_counts_full) + 1), color='lightblue', edgecolor='black')
plt.title('Distribution of Token Counts in the Processed Dataset')
plt.xlabel('Token Count')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

## Push to HuggingFace

In [ ]:
username = "KumudithaSilva"
lite = f"{username}/stream_items_llm_lite"

In [ ]:
train = stream_llm_train
val = stream_llm_val
test = stream_llm_test

Stream_Item.push_to_hub(lite, train, val, test)